# STAT 8017B Project 4 — Data Preprocessing
## Financial Analysis Chatbot (Group 4.1)

This notebook preprocesses all 8 downloaded datasets into clean, model-ready files saved to `data/processed/`.

**Techniques used (from course):**
- NLTK: tokenization, stopword removal, Porter stemming
- scikit-learn: TF-IDF vectorization, StandardScaler, LabelEncoder
- pandas/numpy: missing value handling, type conversion, feature engineering

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib

warnings.filterwarnings('ignore')
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

DATA_DIR = os.path.join('..', 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Setup complete. Output directory:', os.path.abspath(PROCESSED_DIR))

In [ ]:
# Shared text preprocessing function (Course Ch.2 — NLTK pipeline)
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Tokenize, lowercase, remove stopwords, apply Porter stemming."""
    tokens = nltk.word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

print('Text cleaning function ready.')
print('Example:', clean_text('The stock market showed strong gains today in technology sector.'))

---
## 1. ETFs + Mutual Funds
**Source:** `alpha-insights/ETFs.csv`, `MutualFunds.csv`

Select key columns, handle missing values, scale numerical features.

In [ ]:
# --- 1a. ETFs ---
etf_raw = pd.read_csv(os.path.join(DATA_DIR, 'alpha-insights', 'ETFs.csv'))
print(f'ETFs raw: {etf_raw.shape[0]} rows, {etf_raw.shape[1]} columns')

etf_cols = [
    'fund_symbol', 'fund_short_name', 'fund_long_name', 'fund_category', 'fund_family',
    'total_net_assets', 'fund_yield',
    'fund_annual_report_net_expense_ratio',
    'fund_return_ytd', 'fund_return_1year', 'fund_return_3years',
    'fund_return_5years', 'fund_return_10years',
    'fund_mean_annual_return_3years', 'fund_stdev_3years',
    'fund_sharpe_ratio_3years', 'fund_beta_3years',
    'asset_stocks', 'asset_bonds',
    'investment_strategy', 'investment_type', 'size_type'
]
etf = etf_raw[[c for c in etf_cols if c in etf_raw.columns]].copy()

return_cols = ['fund_return_ytd', 'fund_return_1year', 'fund_return_3years',
               'fund_return_5years', 'fund_return_10years']
etf = etf.dropna(subset=return_cols, how='all')

numeric_cols = etf.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    etf[col] = etf[col].fillna(etf[col].median())

etf['investment_strategy'] = etf['investment_strategy'].fillna('')
etf['fund_category'] = etf['fund_category'].fillna('Unknown')
etf['fund_family'] = etf['fund_family'].fillna('Unknown')

# StandardScaler on numerical features (Course Ch.5)
scaler_etf = StandardScaler()
etf_scaled_vals = scaler_etf.fit_transform(etf[numeric_cols])
etf_scaled = etf.copy()
etf_scaled[numeric_cols] = etf_scaled_vals

etf.to_csv(os.path.join(PROCESSED_DIR, 'etf_clean.csv'), index=False)
joblib.dump(scaler_etf, os.path.join(PROCESSED_DIR, 'etf_scaler.joblib'))
print(f'ETFs cleaned: {etf.shape[0]} rows, {etf.shape[1]} columns')
print(f'Numeric columns scaled: {numeric_cols}')
etf.head()

In [ ]:
# --- 1b. Mutual Funds ---
mf_raw = pd.read_csv(os.path.join(DATA_DIR, 'alpha-insights', 'MutualFunds.csv'))
print(f'Mutual Funds raw: {mf_raw.shape[0]} rows, {mf_raw.shape[1]} columns')

mf_cols = [
    'fund_symbol', 'fund_short_name', 'fund_long_name', 'fund_category', 'fund_family',
    'total_net_assets', 'fund_yield',
    'fund_annual_report_net_expense_ratio',
    'fund_return_ytd', 'fund_return_1year', 'fund_return_3years',
    'fund_return_5years', 'fund_return_10years',
    'fund_mean_annual_return_3years', 'fund_stdev_3years',
    'fund_sharpe_ratio_3years', 'fund_beta_3years',
    'investment_strategy', 'investment_type', 'size_type'
]
mf = mf_raw[[c for c in mf_cols if c in mf_raw.columns]].copy()

mf = mf.dropna(subset=return_cols, how='all')

mf_numeric = mf.select_dtypes(include=[np.number]).columns.tolist()
for col in mf_numeric:
    mf[col] = mf[col].fillna(mf[col].median())

mf['investment_strategy'] = mf['investment_strategy'].fillna('')
mf['fund_category'] = mf['fund_category'].fillna('Unknown')
mf['fund_family'] = mf['fund_family'].fillna('Unknown')

mf.to_csv(os.path.join(PROCESSED_DIR, 'mutualfund_clean.csv'), index=False)
print(f'Mutual Funds cleaned: {mf.shape[0]} rows, {mf.shape[1]} columns')
mf.head()

---
## 2. Customer Complaints
**Source:** `complaints/customer_complaints.csv`

Sample 200K rows (stratified by Product), NLTK text preprocessing, TF-IDF vectorization, label encode Product.

In [ ]:
complaints_raw = pd.read_csv(os.path.join(DATA_DIR, 'complaints', 'customer_complaints.csv'))
# CFPB / HF mirrors use snake_case; Kaggle export uses title-case
_cf = {'company': 'Company', 'state': 'State', 'date_received': 'Date received'}
complaints_raw = complaints_raw.rename(columns={k: v for k, v in _cf.items() if k in complaints_raw.columns})
print(f'Complaints raw: {complaints_raw.shape[0]:,} rows')
print(f'Products: {complaints_raw["Product"].nunique()} categories')
print(complaints_raw['Product'].value_counts().head(10))

In [ ]:
SAMPLE_SIZE = 200_000

comp = complaints_raw.dropna(subset=['Product', 'Issue']).copy()

# Stratified sample to preserve product category proportions
if len(comp) > SAMPLE_SIZE:
    comp, _ = train_test_split(
        comp, train_size=SAMPLE_SIZE, random_state=42, stratify=comp['Product']
    )
print(f'Sampled {len(comp):,} rows (stratified by Product)')

comp['text'] = comp['Issue'].fillna('') + ' ' + comp['Sub-issue'].fillna('')
print(f'Applying NLTK text cleaning to {len(comp):,} rows (this will take a few minutes)...')
comp['cleaned_text'] = comp['text'].apply(clean_text)

le_product = LabelEncoder()
comp['product_label'] = le_product.fit_transform(comp['Product'])

tfidf_complaints = TfidfVectorizer(max_features=5000)
X_complaints = tfidf_complaints.fit_transform(comp['cleaned_text'])

comp_save = comp[['Complaint ID', 'Product', 'product_label', 'Issue', 'Sub-issue',
                   'text', 'cleaned_text', 'Company', 'State', 'Date received']].copy()
comp_save.to_csv(os.path.join(PROCESSED_DIR, 'complaints_clean.csv'), index=False)
joblib.dump(tfidf_complaints, os.path.join(PROCESSED_DIR, 'complaints_tfidf.joblib'))
joblib.dump(le_product, os.path.join(PROCESSED_DIR, 'complaints_labels.joblib'))
joblib.dump(X_complaints, os.path.join(PROCESSED_DIR, 'complaints_tfidf_matrix.joblib'))

print(f'Complaints cleaned: {comp_save.shape[0]:,} rows')
print(f'TF-IDF matrix shape: {X_complaints.shape}')
print(f'Product labels: {list(le_product.classes_)}')
comp_save.head()

---
## 3. Financial Q&A Knowledge Base
**Source:** `financial-qa/Financial-QA-10k.csv`

Clean Q&A text, build TF-IDF matrix on questions for cosine-similarity retrieval.

In [ ]:
qa_raw = pd.read_csv(os.path.join(DATA_DIR, 'financial-qa', 'Financial-QA-10k.csv'))
print(f'Financial Q&A raw: {qa_raw.shape[0]:,} rows')
print(f'Columns: {list(qa_raw.columns)}')

qa = qa_raw.dropna(subset=['question', 'answer']).copy()
qa['cleaned_question'] = qa['question'].apply(clean_text)
qa['cleaned_answer'] = qa['answer'].apply(clean_text)

# Build TF-IDF on questions for retrieval (Course Ch.2 — cosine similarity)
tfidf_qa = TfidfVectorizer(max_features=5000)
qa_tfidf_matrix = tfidf_qa.fit_transform(qa['cleaned_question'])

# Save
qa.to_csv(os.path.join(PROCESSED_DIR, 'qa_clean.csv'), index=False)
joblib.dump(tfidf_qa, os.path.join(PROCESSED_DIR, 'qa_tfidf_vectorizer.joblib'))
joblib.dump(qa_tfidf_matrix, os.path.join(PROCESSED_DIR, 'qa_tfidf_matrix.joblib'))

print(f'Q&A cleaned: {qa.shape[0]:,} rows')
print(f'TF-IDF matrix shape: {qa_tfidf_matrix.shape}')
print(f'Tickers covered: {qa["ticker"].nunique()}')
qa[['question', 'answer', 'ticker']].head()

In [ ]:
# Quick retrieval test: cosine similarity (Course Ch.2)
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_answer(query, top_k=3):
    """Find the top-k most similar Q&A pairs for a given query."""
    query_vec = tfidf_qa.transform([clean_text(query)])
    similarities = cosine_similarity(query_vec, qa_tfidf_matrix).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    results = []
    for idx in top_indices:
        results.append({
            'score': round(similarities[idx], 4),
            'question': qa.iloc[idx]['question'],
            'answer': qa.iloc[idx]['answer'][:200] + '...'
        })
    return results

test_results = retrieve_answer('What is the company revenue?')
for r in test_results:
    print(f"Score: {r['score']}")
    print(f"Q: {r['question'][:100]}")
    print(f"A: {r['answer'][:150]}")
    print('---')

---
## 4. Financial Phrasebank (Sentiment)
**Source:** `phrasebank/all-data.csv`

Parse sentiment labels, NLTK text preprocessing, train/test split.

In [ ]:
pb_raw = pd.read_csv(
    os.path.join(DATA_DIR, 'phrasebank', 'all-data.csv'),
    encoding='latin-1', header=None, names=['sentiment', 'sentence']
)
print(f'Phrasebank raw: {pb_raw.shape[0]:,} rows')
print(pb_raw['sentiment'].value_counts())

pb = pb_raw.dropna().copy()
pb['cleaned_text'] = pb['sentence'].apply(clean_text)

# Encode sentiment labels
le_sentiment = LabelEncoder()
pb['sentiment_label'] = le_sentiment.fit_transform(pb['sentiment'])

# Train/test split (80/20 stratified) — Course Ch.3
pb_train, pb_test = train_test_split(pb, test_size=0.2, random_state=42,
                                      stratify=pb['sentiment_label'])

pb.to_csv(os.path.join(PROCESSED_DIR, 'phrasebank_clean.csv'), index=False)
joblib.dump(le_sentiment, os.path.join(PROCESSED_DIR, 'phrasebank_label_encoder.joblib'))

print(f'Phrasebank cleaned: {pb.shape[0]:,} rows')
print(f'Train: {len(pb_train):,}, Test: {len(pb_test):,}')
print(f'Labels: {list(le_sentiment.classes_)}')
pb.head()

---
## 5. FinSen Financial Sentiment
**Source:** `finsen/FinSen_US_Categorized.csv`

NLTK preprocessing on Content, encode Category for topic classification.

In [ ]:
finsen_raw = pd.read_csv(os.path.join(DATA_DIR, 'finsen', 'FinSen_US_Categorized.csv'))
print(f'FinSen raw: {finsen_raw.shape[0]:,} rows')
print(f'Categories: {finsen_raw["Category"].nunique()}')
print(finsen_raw['Category'].value_counts())

finsen = finsen_raw.dropna(subset=['Content', 'Category']).copy()
print(f'\nApplying NLTK text cleaning to {len(finsen):,} rows...')
finsen['cleaned_content'] = finsen['Content'].apply(clean_text)

# Label encode Category for classification
le_finsen = LabelEncoder()
finsen['category_label'] = le_finsen.fit_transform(finsen['Category'])

# TF-IDF on content
tfidf_finsen = TfidfVectorizer(max_features=5000)
X_finsen = tfidf_finsen.fit_transform(finsen['cleaned_content'])

finsen.to_csv(os.path.join(PROCESSED_DIR, 'finsen_clean.csv'), index=False)
joblib.dump(tfidf_finsen, os.path.join(PROCESSED_DIR, 'finsen_tfidf.joblib'))
joblib.dump(le_finsen, os.path.join(PROCESSED_DIR, 'finsen_label_encoder.joblib'))
joblib.dump(X_finsen, os.path.join(PROCESSED_DIR, 'finsen_tfidf_matrix.joblib'))

print(f'FinSen cleaned: {finsen.shape[0]:,} rows')
print(f'TF-IDF matrix: {X_finsen.shape}')
print(f'Category labels: {list(le_finsen.classes_)}')
finsen.head()

---
## 6. Financial News
**Source:** `financial-news/financial_news_events.csv`

Drop empty headlines, parse dates, NLTK preprocessing.

In [ ]:
news_raw = pd.read_csv(os.path.join(DATA_DIR, 'financial-news', 'financial_news_events.csv'))
print(f'Financial News raw: {news_raw.shape[0]:,} rows')
print(f'Columns: {list(news_raw.columns)}')

news = news_raw.dropna(subset=['Headline']).copy()
news['Date'] = pd.to_datetime(news['Date'], errors='coerce')
news['cleaned_headline'] = news['Headline'].apply(clean_text)

# Fill missing Sentiment as 'Unknown' (can predict later with trained model)
news['Sentiment'] = news['Sentiment'].fillna('Unknown')
news['Sector'] = news['Sector'].fillna('Unknown')
news['Impact_Level'] = news['Impact_Level'].fillna('Unknown')

# Convert Index_Change_Percent to numeric
news['Index_Change_Percent'] = pd.to_numeric(news['Index_Change_Percent'], errors='coerce')

news.to_csv(os.path.join(PROCESSED_DIR, 'news_clean.csv'), index=False)
print(f'News cleaned: {news.shape[0]:,} rows')
print(f'Sentiment distribution:\n{news["Sentiment"].value_counts()}')
news.head()

---
## 7. Finance Survey Data
**Source:** `finance-data/Finance_data.csv`

Light cleaning, encode categorical columns.

In [ ]:
survey_raw = pd.read_csv(os.path.join(DATA_DIR, 'finance-data', 'Finance_data.csv'))
print(f'Survey raw: {survey_raw.shape[0]} rows, {survey_raw.shape[1]} columns')

survey = survey_raw.copy()

# Encode categorical columns
cat_cols = survey.select_dtypes(include=['object']).columns.tolist()
label_encoders_survey = {}
for col in cat_cols:
    le = LabelEncoder()
    survey[col + '_encoded'] = le.fit_transform(survey[col].fillna('Unknown'))
    label_encoders_survey[col] = le

survey.to_csv(os.path.join(PROCESSED_DIR, 'survey_clean.csv'), index=False)
joblib.dump(label_encoders_survey, os.path.join(PROCESSED_DIR, 'survey_encoders.joblib'))
print(f'Survey cleaned: {survey.shape[0]} rows, {survey.shape[1]} columns')
print(f'Encoded columns: {cat_cols}')
survey.head()

---
## 8. S&P 500 / ETF / Crypto Prices
**Source:** `sp500-etf-crypto/` (~2,767 individual CSV files)

Scan ALL CSV files in the directory, skip empty ones, merge, compute daily returns and moving averages.

In [ ]:
PRICE_DIR = os.path.join(DATA_DIR, 'sp500-etf-crypto')
MIN_FILE_BYTES = 100  # skip header-only / empty CSVs (42 bytes)

csv_files = sorted([f for f in os.listdir(PRICE_DIR) if f.endswith('.csv')])
print(f'Found {len(csv_files)} CSV files in {PRICE_DIR}')

frames = []
skipped = 0
for fname in csv_files:
    fpath = os.path.join(PRICE_DIR, fname)
    if os.path.getsize(fpath) < MIN_FILE_BYTES:
        skipped += 1
        continue
    try:
        df_t = pd.read_csv(fpath)
        if len(df_t) == 0:
            skipped += 1
            continue
        df_t['ticker'] = fname.replace('.csv', '')
        frames.append(df_t)
    except Exception:
        skipped += 1

prices = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(frames):,} tickers, skipped {skipped} empty/bad files')
print(f'Total rows: {prices.shape[0]:,}')

In [ ]:
prices['Date'] = pd.to_datetime(prices['Date'], errors='coerce')
prices = prices.sort_values(['ticker', 'Date']).reset_index(drop=True)

price_col = 'Close' if 'Close' in prices.columns else 'Adj Close'

prices['daily_return'] = prices.groupby('ticker')[price_col].pct_change()

prices['ma_20'] = prices.groupby('ticker')[price_col].transform(
    lambda x: x.rolling(window=20, min_periods=1).mean()
)
prices['ma_50'] = prices.groupby('ticker')[price_col].transform(
    lambda x: x.rolling(window=50, min_periods=1).mean()
)

fill_cols = [c for c in prices.columns if c not in ['ticker', 'Date']]
prices[fill_cols] = prices.groupby('ticker')[fill_cols].ffill()

prices.to_csv(os.path.join(PROCESSED_DIR, 'prices_clean.csv'), index=False)
print(f'Prices cleaned: {prices.shape[0]:,} rows, {prices.shape[1]} columns')
print(f'Tickers: {prices["ticker"].nunique():,}')
print(f'Date range: {prices["Date"].min()} to {prices["Date"].max()}')
prices.head(10)

---
## Summary of Preprocessed Files
List all files saved to `data/processed/` with their sizes.

In [ ]:
print('=' * 60)
print('PREPROCESSING COMPLETE — Output files:')
print('=' * 60)
total_size = 0
for f in sorted(os.listdir(PROCESSED_DIR)):
    fpath = os.path.join(PROCESSED_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    total_size += size_mb
    print(f'  {f:45s} {size_mb:8.1f} MB')
print('-' * 60)
print(f'  {"TOTAL":45s} {total_size:8.1f} MB')
print('=' * 60)